<a href="https://colab.research.google.com/github/lillymitch/organized_archives/blob/main/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

DATA_URL = "https://media.githubusercontent.com/media/metmuseum/openaccess/master/MetObjects.csv"

df = pd.read_csv(DATA_URL, low_memory=False)

df = df[["Object ID", "Medium", ]]
df = df[df["Medium"].notna()]
df = df[df["Medium"].str.strip().str.lower() != ""]

medium_counts = df["Medium"].value_counts()

top_12_media = medium_counts.head(12).index
df = df[df["Medium"].isin(top_20_media)]

In [ ]:
# balance dataset
filtered_df = (
  df.groupby("Medium")
    .apply(lambda x: x.sample(min(len(x), 300), random_state=42))
    .reset_index(drop=True)
)

In [ ]:
import requests
# fetch image urls
def get_image_url(object_id):
  url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"

  try:
    r = requests.get(url, timeout=5)
    r.raise_for_status()
    data = r.json()
    return data.get("primaryImageSmall") or data.get("primaryImage")

  except:
    return None

from concurrent.futures import ThreadPoolExecutor

def fetch_image_urls(object_ids, max_workers=10):
  with ThreadPoolExecutor(max_workers=max_workers) as executor:
    return list(executor.map(get_image_url, object_ids))

df["image_url"] = fetch_image_urls(df["Object ID"].tolist())
df = df[df["image_url"].notna() & (df["image_url"] != "")]

In [ ]:
# label encoding
unique_media = filtered_df["Medium"].unique()
label_map = {medium: i for i, medium in enumerate(unique_media)}
filtered_df["label"] = filtered_df["Medium"].map(label_map)

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import requests
from io import BytesIO
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

class MetDataset(Dataset):
  def __init__(self, df):
    self.df = df.reset_index(drop=True)

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]

    url = row["image_url"]
    label = row["label"]

    try:
      r = requests.get(url, timeout=5)
      img = Image.open(BytesIO(r.content)).convert("RGB")
      img = transform(img)
    except:
      img = torch.zeros((3, 224, 224))

    return img, torch.tensor(label)

dataset = MetDataset(filtered_df)

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
  dataset,
  [train_size, val_size],
  generator=torch.Generator().manual_seed(42)
)


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
###---------NEW EDIT-----------####
###_____IMAGE PROCESSOR________####
# training: preserves + varies texture

train_processor = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# validation/test: stable, no randomness
test_processor = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
## new edit: ResNet18 transfer learning
from torchvision import models
import torch.nn as nn

def build_transfer_model(num_classes):
  model = models.resnet18(pretrained=True)

  # freeze most layers
  for param in model.parameters():
    param.requires_grad = False

  # unfreeze last layer block (for better performance)
  for param in model.layer4.parameters():
    param.requires_grad = True

  # replace final layer
  model.fc = nn.Linear(model.fc.in_features, num_classes)

  return model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Define a simple Logistic Regression model for image data
class LogisticRegression(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Input image is 3x224x224. Flatten to 3 * 224 * 224 features.
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(3 * 224 * 224, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        return self.linear(x)

# Define the cross-entropy loss function
def cross_entropy_loss(outputs, targets):
    return F.cross_entropy(outputs, targets)

In [ ]:
def test_optimizer(model_builder_function, n_epochs=10):
    # num_classes, cross_entropy_loss, and train_loader are
    # expected to be available globally or passed in.

    small_alpha = 1e-6
    large_alpha = 2e-5

    optimizers_config = {
        "Gradient descent": {'lr': small_alpha, 'momentum': 0.0, 'optimizer_class': torch.optim.SGD},
        "Gradient descent (high learning rate)": {'lr': large_alpha, 'momentum': 0.0, 'optimizer_class': torch.optim.SGD},
        "Momentum": {'lr': small_alpha, 'momentum': 0.9, 'optimizer_class': torch.optim.SGD},
        "Momentum (high learning rate)": {'lr': large_alpha, 'momentum': 0.9, 'optimizer_class': torch.optim.SGD},
        "Adam": {'lr': small_alpha, 'optimizer_class': torch.optim.Adam},
        "Adam (high learning rate)": {'lr': large_alpha, 'optimizer_class': torch.optim.Adam},
    }

    losses_dict = {}

    for name, config in optimizers_config.items():
        # Re-initialize the model for each optimizer test to ensure a clean slate
        model = model_builder_function() # Build the model using the provided function

        # Create optimizer instance, passing model.parameters()
        optimizer_params = {'lr': config['lr']}
        if 'momentum' in config:
            optimizer_params['momentum'] = config['momentum']
        optimizer = config['optimizer_class'](model.parameters(), **optimizer_params)

        losses_dict[name] = []
        for epoch in range(n_epochs):
            total_loss = 0
            # Iterate over the train_loader to get batches of images and labels
            for images, labels in train_loader:
                # Forward pass
                y_pred = model(images)
                # Calculate loss
                loss = cross_entropy_loss(y_pred, labels)

                # Backpropagation
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * images.size(0) # Aggregate loss weighted by batch size

            avg_epoch_loss = total_loss / len(train_loader.dataset) # Average loss for the epoch
            losses_dict[name].append(avg_epoch_loss)

    fig, axarr = plt.subplots(1, 3, figsize = (12, 4), sharey=True)
    for name, losses in losses_dict.items():
        if "Gradient descent" in name:
            i = 0
        elif "Momentum" in name:
            i = 1
        else: # Adam
            i = 2

        if "high learning rate" in name:
            label = name.split(" ")[0] + " (high lr)"
            alpha_val = large_alpha
        else:
            label = name.split(" ")[0]
            alpha_val = small_alpha

        axarr[i].plot(losses, label=r"$\alpha = $" + f"{alpha_val:.0e}", linestyle = '-' if "high learning rate" in name else '--')
        axarr[i].set_title(name.split(" ")[0])
        if i == 2: # Only show legend on the last subplot
            axarr[i].legend()

        if i == 0:
            axarr[i].set_ylabel("Loss")
    for ax in axarr:
        ax.set_xlabel("Epoch")
        ax.grid(True)

    plt.suptitle("Training Loss for Different Optimizers")
    plt.tight_layout()
    plt.show()

# Note: This call will now require a model_builder_function, which will be defined in the checklist.
# For example: test_optimizer(build_my_transfer_learning_model, n_epochs=10)